In [1]:
from unsloth.chat_templates import get_chat_template
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

from transformers import AutoTokenizer, GemmaTokenizerFast
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pythainlp.tokenize import word_tokenize
from nltk.corpus import stopwords
from jiwer import cer

import pandas as pd
import numpy as np
from tqdm import tqdm
import sqlite3
import sys
import os

import torch
import string
import json
import re

import faiss
from langchain.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

ROOT_DIR = "/project/lt200304-dipmt/paweekorn"
MODEL_PATH = f"{ROOT_DIR}/models/base/gemma3-4b-it"
ADAPTER_PATH = None

print("GPU count:", torch.cuda.device_count())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 11-17 10:19:20 [__init__.py:241] Automatically detected platform cuda.
GPU count: 1


## Overview

In [40]:
test_df = pd.read_csv('/project/lt200304-dipmt/paweekorn/data/DS01/test_v1.csv')
with open('/project/lt200304-dipmt/paweekorn/data/wipo/WIPO.json', 'r') as f:
    wipo_data = json.load(f)

wipo_data = {int(k): v for k, v in wipo_data.items()}
test_df['WIPO'] = test_df['NAME'].map(wipo_data)

print(test_df.shape)
test_df.head()

(2785, 4)


,NAME,ENG,THA,WIPO
0,41,"Educational services, namely, conducting semin...",บริการด้านศึกษา ได้แก่ ดำเนินการสัมมนา ประชุม ...,Education; providing of training; entertainmen...
1,41,"training of drivers, road service employees, o...",การฝึกอบรมพนักงานขับรถ พนักงานบริการทางถนน ผู้...,Education; providing of training; entertainmen...
2,41,providing online publications in the nature of...,จัดหาสิ่งพิมพ์ออนไลน์ในลักษณะของจดหมายข่าว ประ...,Education; providing of training; entertainmen...
3,41,information with respect to leisure activities...,ให้ข้อมูลเกี่ยวกับกิจกรรมสันทนาการ ให้ข้อมูลเก...,Education; providing of training; entertainmen...
4,41,"planning, arranging and operating of movies, s...",วางแผน จัดการ และดำเนินการภาพยนตร์ การแสดง ละค...,Education; providing of training; entertainmen...


In [3]:
unique_df = pd.read_csv(f"{ROOT_DIR}/data/unique_no_test.csv")

# in-memory save
db = sqlite3.connect(':memory:')
cur = db.cursor()
cur.execute('create virtual table wipo_table using fts5(eng, tha, tokenize="unicode61");')

# bulk index records
cur.executemany('insert into wipo_table (eng, tha) values (?,?);', unique_df[['ENG', 'THA']].to_records(index=False))
db.commit()

## Retrieval

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
with open(f"{ROOT_DIR}/data/prompt/base_th2en.txt", "r") as f:
    instruction = f.read()

**RAG**

In [5]:
retriever = "bge-m3"
embeddings = HuggingFaceEmbeddings(model_name=f"{ROOT_DIR}/models/retriever/{retriever}")
vectorstore = FAISS.load_local(
    f"{ROOT_DIR}/vector/th2en/{retriever}", 
    embeddings,
    allow_dangerous_deserialization=True
)
gpu_index = faiss.index_cpu_to_gpu(faiss.StandardGpuResources(), 0, vectorstore.index)
vectorstore.index = gpu_index

def rag_relevant_docs(embeddings, vectorstore, query, k):
    """return top k most cosine similarity score"""
    query_embedding = embeddings.embed_query(query)
    docs = vectorstore.similarity_search_by_vector(query_embedding, k=k)

    result = "\n## Retrieved References\n"
    for doc in docs:
        result += f"""THA: {doc.page_content}
ENG: {doc.metadata['english']}\n\n"""
    return result

**FTS**

In [6]:
def construct_query(text):
    """construct condition matching for SQL query via fts5"""
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    tokens = word_tokenize(text, engine="attacut")
    tokens = [f'(THA:{t}) OR ' for t in tokens if not (t.isspace() or t=="")]
    return "".join(tokens)[:-3]

def fts_relevant_docs(cur, eng_text, k):
    """bm25 search via fts5 in SQL query"""
    query = construct_query(eng_text)
    res = cur.execute(f"""SELECT ENG, THA, bm25(wipo_table) AS rank
    FROM wipo_table
    WHERE wipo_table MATCH "{query}"
    ORDER BY rank
    LIMIT {k};
    """).fetchall()

    result = "\n## Retrieved References\n"
    for doc in res:
        result += f"""THA: {doc[1]}
ENG: {doc[0]}\n\n"""
    return result

**Construct prompt**

In [8]:
def formatting_prompt(df):
    batch = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = instruction.format(
            WIPO=row['WIPO'],
            # RAG_DOC="",
            RAG_DOC=rag_relevant_docs(embeddings, vectorstore, row['THA'], k=3),
            THAI=row["THA"]
        )
        chat = [{"role": "user", "content": prompt}]
        chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        batch.append(chat)
    return batch

test_set = formatting_prompt(test_df)
print(test_set[0])

100%|██████████| 2785/2785 [00:41<00:00, 67.84it/s]

<bos><start_of_turn>user
## Instructions:
You are an expert in the classification of goods and services under the WIPO Nice Classification system. Your task is to translate product names from Thai to natural and clear English.

**Translation Guidelines:**
- Maintain the punctuation alignment of the input text.
- Use Thai legal and commercial terminology appropriate for trademarks and product classification.
- Do not include explanations, commentary, or any information beyond the translation output.
- Provided output in the format of {"eng_translation": "English translation"}

**Product Domain:**
Education; providing of training; entertainment; sporting and cultural activities.

## Retrieved References
THA: บริการด้านศึกษา, ได้แก่, ดำเนินการสัมมนา, ประชุมเชิงปฏิบัติ, สอนออนไลน์และฝึกอบรมใช้งานคอมพิวเตอร์ในสาขาซอฟต์แวร์คอมพิวเตอร์, วิเคราะห์ธุรกิจ, และระบบธุรกิจอัจฉริยะและจัดจำหน่ายสื่อเรียนสอนในส่วนที่เกี่ยวข้อง
ENG: Educational service, namely, conducting seminar, conference, workshop,

## Inference

In [10]:
if not bool(ADAPTER_PATH) or "fine-tuned" in MODEL_PATH:
    enable_lora = False ; lora_req = None
    tensor_parallel = 1
else:
    enable_lora = True; lora_req = LoRARequest("lora_adapter", 1, ADAPTER_PATH) if ADAPTER_PATH else None;
    tensor_parallel = torch.cuda.device_count()

model = LLM(
    model=MODEL_PATH,
    quantization="bitsandbytes",
    max_model_len=4096,
    tensor_parallel_size=tensor_parallel,
    enable_prefix_caching=True,
    gpu_memory_utilization=0.5,
    enforce_eager=True,
    enable_lora=enable_lora,
    max_lora_rank=64
)

INFO 11-17 10:20:52 [utils.py:326] non-default args: {'model': '/project/lt200304-dipmt/paweekorn/models/base/gemma3-4b-it', 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'max_lora_rank': 64}
INFO 11-17 10:21:01 [__init__.py:711] Resolved architecture: Gemma3ForConditionalGeneration
INFO 11-17 10:21:01 [__init__.py:1750] Using max model len 4096
WARNING 11-17 10:21:01 [__init__.py:1171] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 11-17 10:21:03 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-17 10:21:03 [__init__.py:3565] Cudagraph is disabled under eager mode
WARNING 11-17 10:21:06 [__init__.py:2921] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.h

(EngineCore_0 pid=3712490) Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


(EngineCore_0 pid=3712490) INFO 11-17 10:21:22 [gpu_model_runner.py:1953] Starting to load model /project/lt200304-dipmt/paweekorn/models/base/gemma3-4b-it...
(EngineCore_0 pid=3712490) INFO 11-17 10:21:23 [gpu_model_runner.py:1985] Loading model from scratch...
(EngineCore_0 pid=3712490) INFO 11-17 10:21:23 [cuda.py:345] Using FlexAttention backend for head_size=72 on V1 engine.
(EngineCore_0 pid=3712490) INFO 11-17 10:21:23 [__init__.py:3565] Cudagraph is disabled under eager mode
(EngineCore_0 pid=3712490) INFO 11-17 10:21:23 [cuda.py:328] Using Flash Attention backend on V1 engine.
(EngineCore_0 pid=3712490) INFO 11-17 10:21:23 [bitsandbytes_loader.py:742] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.46s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.70s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.67s/it]
(EngineCore_0 pid=3712490) 


(EngineCore_0 pid=3712490) INFO 11-17 10:21:29 [gpu_model_runner.py:2007] Model loading took 3.7575 GiB and 5.869000 seconds
(EngineCore_0 pid=3712490) INFO 11-17 10:21:29 [gpu_model_runner.py:2591] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 31 image items of the maximum feature size.
(EngineCore_0 pid=3712490) INFO 11-17 10:21:33 [gpu_worker.py:276] Available KV cache memory: 12.66 GiB
(EngineCore_0 pid=3712490) WARNING 11-17 10:21:33 [kv_cache_utils.py:971] Add 1 padding layers, may waste at most 3.45% KV cache memory
(EngineCore_0 pid=3712490) INFO 11-17 10:21:33 [kv_cache_utils.py:1013] GPU KV cache size: 94,800 tokens
(EngineCore_0 pid=3712490) INFO 11-17 10:21:33 [kv_cache_utils.py:1017] Maximum concurrency for 4,096 tokens per request: 23.07x
(EngineCore_0 pid=3712490) INFO 11-17 10:21:34 [core.py:214] init engine (profile, create kv cache, warmup model) took 4.74 seconds
(EngineCore_0 pid=3712490) INFO 11-17 10:21:35 [__init__.py:3565] Cud

In [11]:
decoding_params = SamplingParams(
    temperature=0.0, top_p=1, top_k=-1,
    max_tokens=4096,
    skip_special_tokens=True,
    repetition_penalty=1.15,
    frequency_penalty=0.2,
)

results = model.generate(test_set, decoding_params, lora_request=lora_req)
response = [r.outputs[0].text for r in results]
response[0]

Adding requests:   0%|          | 0/2785 [00:00<?, ?it/s]

(EngineCore_0 pid=3712490) WARNING 11-17 10:21:35 [cudagraph_dispatcher.py:101] cudagraph dispatching keys are not initialized. No cudagraph will be used.


Processed prompts:   0%|          | 0/2785 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(EngineCore_0 pid=3712490) [rank0]:W1117 10:22:46.872000 3712490 /lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:964] [0/8] torch._dynamo hit config.recompile_limit (8)
(EngineCore_0 pid=3712490) [rank0]:W1117 10:22:46.872000 3712490 /lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    function: 'forward_static' (/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packages/vllm/model_executor/layers/layernorm.py:221)
(EngineCore_0 pid=3712490) [rank0]:W1117 10:22:46.872000 3712490 /lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    last reason: 0/7: expected type of 'residual' to be a tensor type, ' but found <class 'NoneType'>
(EngineCore_0 pid=3712490) [rank0]:W1117 10:22:46.872000 3712490 /lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packa

'{"eng_translation": "Educational service, namely, conducting seminars, conferences, workshops, specialized online tutoring and computer usage training in the area of computer software, business analysis, and intelligent business systems, and distributing related teaching materials."}\n'

## Evaluation

In [14]:
def filter_english(text):
    text = re.sub(r'[^a-zA-Z\,\(\)\[\]\s]', '', text)
    return text.replace("\n", "")

def extract_json(text):
    text = text[text.rfind("{"):]
    pattern = r'''{\s*[\'\"]eng_translation[\'\"]:\s*[\'\"].*?[\'\"]\s*}'''
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        try:
            loaded = json.loads(matches[0])
            return loaded['eng_translation']
        except json.JSONDecodeError as e:
            return filter_english(text)
    else:
        return filter_english(text)


test_df['PRED'] = response
test_df['PRED_cleaned'] = test_df['PRED'].apply(extract_json)
print("Null rows: ", test_df['PRED_cleaned'].isnull().sum())
test_df.head()

Null rows:  0


,NAME,ENG,THA,WIPO,PRED,PRED_cleaned
0,41,"Educational services, namely, conducting semin...",บริการด้านศึกษา ได้แก่ ดำเนินการสัมมนา ประชุม ...,Education; providing of training; entertainmen...,"{""eng_translation"": ""Educational service, name...","Educational service, namely, conducting semina..."
1,41,"training of drivers, road service employees, o...",การฝึกอบรมพนักงานขับรถ พนักงานบริการทางถนน ผู้...,Education; providing of training; entertainmen...,"{""eng_translation"": ""training of driver, road ...","training of driver, road service employee, ope..."
2,41,providing online publications in the nature of...,จัดหาสิ่งพิมพ์ออนไลน์ในลักษณะของจดหมายข่าว ประ...,Education; providing of training; entertainmen...,"{""eng_translation"": ""online provision of newsl...","online provision of newsletters, announcements..."
3,41,information with respect to leisure activities...,ให้ข้อมูลเกี่ยวกับกิจกรรมสันทนาการ ให้ข้อมูลเก...,Education; providing of training; entertainmen...,"{""eng_translation"": ""information regarding rec...","information regarding recreational activity, i..."
4,41,"planning, arranging and operating of movies, s...",วางแผน จัดการ และดำเนินการภาพยนตร์ การแสดง ละค...,Education; providing of training; entertainmen...,"{""eng_translation"": ""planning, arranging and o...","planning, arranging and operating of movies, s..."


In [50]:
cer_result, bleu = [], []
chencherry = SmoothingFunction().method1
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    sol = row["ENG"]
    pred = "" if pd.isna(row["PRED_cleaned"]) else row["PRED_cleaned"]

    # bleu score
    ref = word_tokenize(sol, engine="attacut")
    hyp = word_tokenize(pred, engine="attacut")

    ref = [word for word in ref if not word.isspace()]
    hyp = [word for word in hyp if not word.isspace()]
    
    cer_result.append(cer(sol, pred))
    bleu.append(sentence_bleu([ref], hyp, smoothing_function=chencherry))

test_df["BLEU"] = bleu
print(f"Average CER:", np.mean(cer_result).round(4))
print(f"Average BLEU:", np.mean(bleu).round(4))

100%|██████████| 2785/2785 [00:24<00:00, 111.89it/s]

Average CER: 0.4335
Average BLEU: 0.1516


In [16]:
MODEL_ID = MODEL_PATH.split('/')[-1] if "checkpoint" not in MODEL_PATH else MODEL_PATH.split('/')[-2]
fname = f"{MODEL_ID}_base+RAG"
save_path = f"{ROOT_DIR}/data/infer-result/th2en/{fname}.csv"
test_df[['PRED', 'PRED_cleaned']].to_csv(save_path, index=False)
print(f"Save at {save_path} already!")

Save at /project/lt200304-dipmt/paweekorn/data/infer-result/th2en/gemma3-4b-it_base+RAG.csv already!
